In [0]:
  # Databricks notebook source

# ============================================================
# 01_BRONZE
# Pipeline de Dados - Inside Airbnb São Paulo
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
 # ============================================================
# VERIFICAR ARQUIVOS DO VOLUME
# ============================================================

display(
    dbutils.fs.ls("/Volumes/workspace/default/airbnb_sp")
)

In [0]:
 display(
    dbutils.fs.ls("/Volumes/workspace/default/airbnb_sp/raw")
)

In [0]:
 # ============================================================
# CONFIGURAÇÃO
# ============================================================

RAW_PATH = "/Volumes/workspace/default/airbnb_sp/raw/listings.csv.gz"

BRONZE_TABLE = "workspace.default.bronze_listings"

print("RAW:", RAW_PATH)
print("BRONZE:", BRONZE_TABLE)

In [0]:
 # ============================================================
# INGESTÃO DO RAW
# ============================================================

RAW_PATH = "/Volumes/workspace/default/airbnb_sp/raw/listings.csv.gz"

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv(RAW_PATH)
)

print("Linhas:", df_raw.count())
print("Colunas:", len(df_raw.columns))

display(df_raw.limit(10))

In [0]:
 # COMMAND ----------

# ============================================================
# AUDITORIA INICIAL
# ============================================================

print("  de registros:", df_raw.count())
print("Quantidade de atributos:", len(df_raw.columns))

display(
    df_raw.select(
        F.count("*").alias("total_registros")
    )
)

In [0]:
 # COMMAND ----------

# ============================================================
# INSPEÇÃO DO ESQUEMA
# ============================================================

df_raw.printSchema()

In [0]:
 # COMMAND ----------

# ============================================================
# CONTROLE DE DUPLICIDADE DO ID
# ============================================================

if "id" in df_raw.columns:

    total_ids = df_raw.select("id").count()

    ids_distintos = df_raw.select("id").distinct().count()

    duplicados = total_ids - ids_distintos

    print("Total de IDs:", total_ids)
    print("IDs distintos:", ids_distintos)
    print("IDs duplicados:", duplicados)

else:
    print("Coluna 'id' não encontrada.")

In [0]:
 # ============================================================
# PERSISTÊNCIA BRONZE
# ============================================================

(
    df_raw.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Tabela Bronze criada/atualizada: {BRONZE_TABLE}")

In [0]:
 # COMMAND ----------

# ============================================================
# VALIDAÇÃO DA BRONZE
# ============================================================

df_bronze = spark.table(BRONZE_TABLE)

print("Registros Bronze:", df_bronze.count())
print("Atributos Bronze:", len(df_bronze.columns))

display(df_bronze.limit(10))